# Advanced Feature Engineering
In this exercise we will learn about the importance of feature engineering to improve our model's performance.
We will be working on a kaggle dataset of kickstarter projects (if you don't know Kickstarter (shame on you!), your first assignment is to visit the <a href='https://www.kickstarter.com/'>Kickstarter</a> website and find a cool project).
Each record represents one project and some basic information about it. 

The dataset can be found <a href='https://www.kaggle.com/kemical/kickstarter-projects#'> HERE </a>

In this exercise we will try to predict whether a project will be a success or not (binary classification).

Have fun :)

``` ~Lior Hirsch ```

```First, make sure all the following libraries are installed on you computer.```

In [39]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
from sklearn.preprocessing import OrdinalEncoder
import category_encoders as ce
import itertools
from currency_converter import CurrencyConverter
import xgboost as xgb

In [40]:
random_state = 1
np.random.seed(random_state)

## Loading and cleaning the data
```The following code is a simple cleaning of the dataset. In the original dataset there are several states (which is the target column). Instead, we will use a new generated binary column based on the state column - output.```

```Your first assignment is to create a train-test-split that will fit to our dataset.```

#### Questions

```How can you test whether the train-test-split is good?```

```Explain your train-test-split proposal and why does it fit the dataset?```

In [41]:
def split(data):
    ks_train, ks_test = train_test_split(data, test_size=0.3, random_state=random_state)
    return ks_train, ks_test

Answers:
1. Ensure there's no form of snooping and check if the train error and test error are at least somewhat close

What I mean by snooping: we don't want to use things in the test during training. This may not be as easy as simply dividing the dataset, because the samples might not be completely independent; As an example take graphical data represented by an adjacency list, a row being in the test does not guarantee we haven't seen its node in the train as part of the neighbors of some other node.

What I mean by somewhat close: if the train error is large but the test error is small we might have just picked too small of a test set and got lucky on it.

2. The dataset is large and its rows are independent so a standard split should work.

Different ways we could have split the data: by forms of grouping, e.g. if there were sequel projects it would probably be a good idea to make sure both of them are either in the train or the test

In [42]:
def load_clean_split_datasets():
    ks = pd.read_csv('ks-projects-201801.csv',
                 parse_dates=['deadline', 'launched'])
    # Drop live projects
    ks = ks[ks.state != 'live']

    # Add outcome column, "successful" == 1, others are 0
    ks['output'] = (ks['state'] == 'successful').astype(int)

    # Drop pledged columns
    ks = ks.drop(columns = ['pledged', 'backers', 'usd pledged', 'usd_pledged_real', 'usd_goal_real', 'state'])
    
    ks_train, ks_test = split(ks)
    ks_train = ks_train.reset_index()
    ks_test = ks_test.reset_index()
    
    return ks_train, ks_test

## Baseline model
```The following code builds a baseline model. Our data contains categorical columns. Therefore we need to encode them (Don't worry, we will learn about different encoders). For now we will use a basic encoding method called LabelEncoder. Read about this encoder. ```

```Pay attention to the helper methods which will be used in this exercise.```

In [43]:
def get_xy_by_columns(ks_train, ks_test, columns):
    x_train = ks_train[columns].copy()
    y_train = ks_train['output'].copy()

    x_test = ks_test[columns].copy()
    y_test = ks_test['output'].copy()
    
    return x_train, x_test, y_train, y_test

In [44]:
def fit_evaluate(x_train, x_val, y_train, y_val):
    cls = xgb.XGBClassifier(n_jobs = -1, n_estimators=50, max_depth = 5, random_state=random_state)
    cls.fit(x_train, y_train)
    
    preds = cls.predict_proba(x_train)
    print(f"AUC of ROC on train : {np.round(roc_auc_score(y_train, preds[:,1]), 4)}")
    
    preds = cls.predict_proba(x_val)
    print(f"AUC of ROC on validation : {np.round(roc_auc_score(y_val, preds[:,1]), 4)}")
    
    return cls

In [45]:
ks_train, ks_test = load_clean_split_datasets()
relevant_columns = ['category', 'main_category', 'currency', 'goal', 'country']
x_train, x_test, y_train, y_test = get_xy_by_columns(ks_train, ks_test, relevant_columns)

```Before fitting a model to our data, we need to encode the categorical data to a numeric/float types (Why?). LabelEncoder is a simple encoding method. Read about this encoder and use it.```

```You might find categories in the test set which are not exist in the train set. Think how to fix this problem.```

In [46]:
def label_encode(x_train, x_test, cols, handle_unknown='value', most_frequent=False):
    oe = ce.ordinal.OrdinalEncoder(handle_unknown=handle_unknown, cols=cols)
    oe.fit(x_train)
    x_train = oe.transform(x_train)
    x_test = oe.transform(x_test)
    if most_frequent:
        for col in cols:
            mf = x_train[col].mode()[0]
            x_test[col] = x_test[col].replace(-1, mf)
    return x_train, x_test
x_train, x_test = label_encode(x_train, x_test, ['category', 'main_category', 'currency', 'country'], most_frequent=True)

Another way to handle unknown categories is to assign them the most frequent category

Cons of allowing new categories and assigning -1:

This gives -1 actual meaning and could affect how the model percieves its importance instead of it being arbitrary, especially since everything else is $\geq 0$. Specifically with xgb (or other tree bases methods) this might also make the model incorrectly classify it as being the same as other categories, for example if we had a $\geq 1$ step it would normally be a seperator of 0 and everything else, whereas now it would include -1.

In [47]:
fit_evaluate(x_train, x_test, y_train, y_test)

AUC of ROC on train : 0.7317
AUC of ROC on validation : 0.7246


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

## Date encoding
```First, we'll start with a basic date encoding for the launch date and the deadline date. For each date, create three new columns - the hour, day and the month of the date. At the end of this encoding the data should contain six new columns - launched_hour, launched_day, launched_month, deadline_hour, deadline_day, deadline_month ```

#### Questions
```Why we won't create a column for the year?```

Answer:
Year does not repeat whereas hour, day, month do (periodically)

In [48]:
def encode_launch_dt(df):
    df["launched_hour"] = df["launched"].dt.hour
    df["launched_day"] = df["launched"].dt.day
    df["launched_month"] = df["launched"].dt.month
    return df

def encode_deadline_dt(df):
    df["deadline_hour"] = df["deadline"].dt.hour
    df["deadline_day"] = df["deadline"].dt.day
    df["deadline_month"] = df["deadline"].dt.month
    return df

In [49]:
ks_train, ks_test = load_clean_split_datasets()

ks_train = encode_launch_dt(ks_train)
ks_train = encode_deadline_dt(ks_train)

ks_test = encode_launch_dt(ks_test)
ks_test = encode_deadline_dt(ks_test)

In [50]:
relevant_columns = ['category', 'main_category', 'currency', 'goal', 'country',
                     'launched_hour', 'launched_day', 'launched_month',
                     'deadline_hour', 'deadline_day', 'deadline_month']

x_train, x_test, y_train, y_test = get_xy_by_columns(ks_train, ks_test, relevant_columns)

```Use the label encoder you used in the previous section to encode the categorical columns.```

In [51]:
x_train, x_test = label_encode(x_train, x_test, ['category', 'main_category', 'currency', 'country'])

In [52]:
cls = fit_evaluate(x_train, x_test, y_train, y_test)

AUC of ROC on train : 0.7451
AUC of ROC on validation : 0.7349


## Categorical Encoding
```Next, we will learn about different categorical encodings. Read how each encoding method works and try to understand when we should use each one of them. Be ready to discuss this with your tutor```

```You can start by reading the following blog-posts:```

https://wrosinski.github.io/fe_categorical_encoding/

https://www.analyticsvidhya.com/blog/2020/08/types-of-categorical-data-encoding/

### Count encoding

In [53]:
def count_encoding(x_train, x_val, col):
    coe = ce.count.CountEncoder(cols=[col], handle_unknown='value')
    coe.fit(x_train)
    x_train = coe.transform(x_train)
    x_val = coe.transform(x_val)
    return (x_train, x_val)
        

In [54]:
ks_train, ks_test = load_clean_split_datasets()

ks_train = encode_launch_dt(ks_train)
ks_train = encode_deadline_dt(ks_train)

ks_test = encode_launch_dt(ks_test)
ks_test = encode_deadline_dt(ks_test)

relevant_columns = ['category', 'main_category', 'currency', 'goal', 'country',
                     'launched_hour', 'launched_day', 'launched_month',
                     'deadline_hour', 'deadline_day', 'deadline_month']

x_train, x_test, y_train, y_test = get_xy_by_columns(ks_train, ks_test, relevant_columns)

In [55]:
cat_features = ['category', 'main_category', 'currency', 'country']

for curr_cat in cat_features:
    x_train, x_test = count_encoding(x_train, x_test, curr_cat)
    
x_test = x_test.fillna(0)

print(x_train.head())

'''
why was this here?
x_train = x_train.drop(columns=cat_features)
x_test = x_test.drop(columns=cat_features)
'''

   category  main_category  currency     goal  country  launched_hour  \
0     11252          44138    205626  25000.0   203691             23   
1      4170          27745    205626   1000.0   203691              5   
2      5749          19488    205626   7500.0   203691             11   
3      2883          20727    205626   1000.0   203691             17   
4      5803          27745    205626  75000.0   203691              6   

   launched_day  launched_month  deadline_hour  deadline_day  deadline_month  
0            27               4              0            26               6  
1            29              10              0            28              12  
2            19               2              0            10               4  
3            10              10              0             9              11  
4             2               6              0             2               7  


'\nwhy was this here?\nx_train = x_train.drop(columns=cat_features)\nx_test = x_test.drop(columns=cat_features)\n'

Think about why I added the new columns: I think it's just an implementation difference, the way I wrote count_encoding changes the columns instead of adding new ones so there is no reason to drop them. 

In [56]:
fit_evaluate(x_train, x_test, y_train, y_test)

AUC of ROC on train : 0.7461
AUC of ROC on validation : 0.7352


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

### Target encoder
```Target encoding can be used with/without smoothing. We will implement both types. We'll start without smoothing. Read about the difference between them.```

In [57]:
def calc_smooth_mean(df, cat_col, target, smoothing = 0):
    n = df.value_counts(cat_col).sum()
    w = n / (smoothing + n)
    mean = df[target].mean()
    return (w * df.groupby(cat_col)[target].mean() + (1-w) * mean)


In [58]:
ks_train, ks_test = load_clean_split_datasets()

ks_train = encode_launch_dt(ks_train)
ks_train = encode_deadline_dt(ks_train)

ks_test = encode_launch_dt(ks_test)
ks_test = encode_deadline_dt(ks_test)

relevant_columns = ['category', 'main_category', 'currency', 'goal', 'country',
                     'launched_hour', 'launched_day', 'launched_month',
                     'deadline_hour', 'deadline_day', 'deadline_month']
ks_train = ks_train.fillna(0)
ks_test = ks_test.fillna(0)

In [59]:

cat_features = ['category', 'main_category', 'currency', 'country']
m = 0
mean = ks_train['output'].mean()

for curr_cat in cat_features:
    smooth_mean = calc_smooth_mean(ks_train, curr_cat, "output", m)
    ks_train[curr_cat] = ks_train[curr_cat].map(smooth_mean)
    ks_test[curr_cat] = ks_test[curr_cat].map(smooth_mean)
    ks_test[curr_cat] = ks_test[curr_cat].fillna(mean)

x_train, x_test, y_train, y_test = get_xy_by_columns(ks_train, ks_test, relevant_columns)

In [60]:
fit_evaluate(x_train, x_test, y_train, y_test)

AUC of ROC on train : 0.7476
AUC of ROC on validation : 0.7372


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [61]:
ks_train, ks_test = load_clean_split_datasets()

ks_train = encode_launch_dt(ks_train)
ks_train = encode_deadline_dt(ks_train)

ks_test = encode_launch_dt(ks_test)
ks_test = encode_deadline_dt(ks_test)

relevant_columns = ['category', 'main_category', 'currency', 'goal', 'country',
                     'launched_hour', 'launched_day', 'launched_month',
                     'deadline_hour', 'deadline_day', 'deadline_month']

In [62]:

cat_features = ['category', 'main_category', 'currency', 'country']
m = 15
mean = ks_train['output'].mean()

for curr_cat in cat_features:
    smooth_mean = calc_smooth_mean(ks_train, curr_cat, "output", m)
    ks_train[curr_cat] = ks_train[curr_cat].map(smooth_mean)
    ks_test[curr_cat] = ks_test[curr_cat].map(smooth_mean)
    ks_test[curr_cat] = ks_test[curr_cat].fillna(mean)

x_train, x_test, y_train, y_test = get_xy_by_columns(ks_train, ks_test, relevant_columns)

Why was $m=100000$: just a leftover from expirementing with different values, changed to a more sensible one. 

In [63]:
fit_evaluate(x_train, x_test, y_train, y_test)

AUC of ROC on train : 0.7476
AUC of ROC on validation : 0.7372


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

### Catboost encoding

In [64]:
# Catboost encoding
ks_train, ks_test = load_clean_split_datasets()

ks_train = encode_launch_dt(ks_train)
ks_train = encode_deadline_dt(ks_train)

ks_test = encode_launch_dt(ks_test)
ks_test = encode_deadline_dt(ks_test)

relevant_columns = ['category', 'main_category', 'currency', 'goal', 'country',
                     'launched_hour', 'launched_day', 'launched_month',
                     'deadline_hour', 'deadline_day', 'deadline_month']
x_train, x_test, y_train, y_test = get_xy_by_columns(ks_train, ks_test, relevant_columns)

In [65]:
def catboost_encode(x_train, x_test):
    cat_features = ['category', 'main_category', 'currency', 'country']
    catboost = ce.cat_boost.CatBoostEncoder(cols=cat_features, handle_unknown='value')
    catboost.fit(x_train, y_train)
    x_train = catboost.transform(x_train)
    x_test = catboost.transform(x_test)
    return x_train, x_test
x_train, x_test = catboost_encode(x_train, x_test)


In [66]:
fit_evaluate(x_train, x_test, y_train, y_test)

AUC of ROC on train : 0.7472
AUC of ROC on validation : 0.7366


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

## Feature Generation
```Creating new features from the raw data is a powerful way to improve your model performance.```

```In the following section we will generate the features and evaluate their impact in the end.```

In [67]:
ks_train, ks_test = load_clean_split_datasets()

ks_train = encode_launch_dt(ks_train)
ks_train = encode_deadline_dt(ks_train)

ks_test = encode_launch_dt(ks_test)
ks_test = encode_deadline_dt(ks_test)

relevant_columns = ['category', 'main_category', 'currency', 'goal', 'country', 'launched', 'deadline',
                     'launched_hour', 'launched_day', 'launched_month',
                     'deadline_hour', 'deadline_day', 'deadline_month']
x_train, x_test, y_train, y_test = get_xy_by_columns(ks_train, ks_test, relevant_columns)

cat_features = ['category', 'main_category', 'currency', 'country']

### Interactions
```One of the easiest ways to create new features is by combining categorical variables. Create all the combinations of any two categoricals features. Don't forget to encode the new categorical features.```

#### Questions
```When is this not a good idea to and how it can be solved?```

Answer:
Not a good idea if there are many categories per column (if we have m and n then this creates mn new ones). To solve this we can take only the columns with few categories

In [68]:
cat_combs = []
visited_cat = [
    [False] * len(cat_features)
for _ in range(len(cat_features))] 
for i, cat1 in enumerate(cat_features):
    for j, cat2 in enumerate(cat_features):
        if not visited_cat[i][j] and not visited_cat[j][i] and i != j:
            visited_cat[i][j] = True
            visited_cat[j][i] = True
            cat_combs.append((cat1, cat2))
for cats in cat_combs:
    cat1, cat2 = cats
    x_train[f"{cat1}_{cat2}"] = ks_train[cat1] + " " + ks_train[cat2]
    x_test[f"{cat1}_{cat2}"] = ks_test[cat1] + " " + ks_test[cat2]

In [69]:
catboost = ce.cat_boost.CatBoostEncoder(cols=cat_features + [x[0]+"_"+x[1] for x in cat_combs], handle_unknown='value')
catboost.fit(x_train, y_train)
x_train = catboost.transform(x_train)
x_test = catboost.transform(x_test)

### Domain knowledge 
```In this section we will create new features based on the original ones with a pinch of imagination and creativity. The domain knowledge features generation is one of the most successful ways to improve our model. ```

```Don't forget to add the new features both to x_train, x_test```

```Create a feature that contains the goal in USD currency```

In [70]:
ks_train, ks_test = load_clean_split_datasets()

ks_train = encode_launch_dt(ks_train)
ks_train = encode_deadline_dt(ks_train)

ks_test = encode_launch_dt(ks_test)
ks_test = encode_deadline_dt(ks_test)

relevant_columns = ['category', 'main_category', 'currency', 'goal', 'country', 'launched', 'deadline',
                     'launched_hour', 'launched_day', 'launched_month',
                     'deadline_hour', 'deadline_day', 'deadline_month']
x_train, x_test, y_train, y_test = get_xy_by_columns(ks_train, ks_test, relevant_columns)
c = CurrencyConverter()
currs = {currency: c.convert(1, currency, 'USD') for currency in x_train['currency'].unique()}

x_train["goal_usd"] = x_train['goal'] * x_train['currency'].map(currs)
x_test["goal_usd"] = x_test['goal'] * x_test['currency'].map(currs)

```Count the number of projects launched in the preceeding week for each record.```

In [71]:
last_week = pd.to_datetime("today") - pd.Timedelta(days=7)
x_train["proj_last_week"] = x_train[last_week <= x_train['launched']].count()
x_test["proj_last_week"] = x_test[last_week <= x_test['launched']].count()

```Count the days each project was online```

In [72]:
x_train["online"] = (x_train["deadline"] - x_train["launched"]).dt.days + 1
x_test["online"] = (x_test["deadline"] - x_test["launched"]).dt.days + 1

```Calculate the goal per day```

In [73]:
x_train["goal_per_day"] = x_train["goal"]/x_train["online"]
x_test["goal_per_day"] = x_test["goal"]/x_test["online"]

```Calculate the goal per day in USD```

In [74]:
x_train["goal_usd_per_day"] = x_train["goal_usd"]/x_train["online"]
x_test["goal_usd_per_day"] = x_test["goal_usd"]/x_test["online"]

```Calculate the time since the last launch project in the same category```

In [75]:
x_train = x_train.sort_values("launched").reset_index()
x_train["time_since_last_launch"] = (x_train["launched"] - x_train.sort_values("launched")["launched"].shift(1)).dt.days
x_test = x_test.sort_values("launched").reset_index()
x_test["time_since_last_launch"] = (x_test["launched"] - x_test.sort_values("launched")["launched"].shift(1)).dt.days

### Transforming numerical features
```Numerical features can be transformed with mathematical transformation like log, sqrt etc. Create another two features - log(goal_usd), sqrt(goal_usd)```

#### Questions
```Why are those transformation useful? ```

```In which cases/ models we should use this transformation?```


Answers:
1. They allow the model to learn non linear relationships. log and $\sqrt{}$ specifically are useful for dealing with extreme values and for generally making the data less variant and more normally distributed

Why: if our values are very skewed towards large numbers than log compresses them heavily while only lightly compressing smaller values, e.g. $[1, 10000] \rightarrow [0, \sim 9.2]$ with $5\rightarrow \sim 1.6$ but $10000\rightarrow \sim 9.2$. This works the same with $\sqrt{}$ to a lesser extent, so it's better suited for stuff that's leaning toward large values but not extremely so. 

2. When values span a very large interval 

In [76]:
x_train["log_goal"] = np.log(x_train["goal_usd"])
x_test["log_goal"] = np.log(x_test["goal_usd"])
x_train["sqrt_goal"] = np.sqrt(x_train["goal_usd"])
x_test["sqrt_goal"] = np.sqrt(x_test["goal_usd"])

In [77]:
x_train = x_train.drop(columns=["launched", "deadline", "proj_last_week"])
x_test = x_test.drop(columns=["launched", "deadline", "proj_last_week"])
cat_features = ['category', 'main_category', 'currency', 'country']
catboost = ce.cat_boost.CatBoostEncoder(cols=cat_features)
catboost.fit(x_train, y_train)
x_train = catboost.transform(x_train)
x_test = catboost.transform(x_test)

In [78]:
x_train = x_train.fillna(0)
x_test = x_test.fillna(0)
cls = fit_evaluate(x_train, x_test, y_train, y_test)

AUC of ROC on train : 0.5765
AUC of ROC on validation : 0.4985


```Great! Now create another five unique and creative features that will make your tutor impressed and improve the model's validation AUC further. ```

In [79]:
ks_train, ks_test = load_clean_split_datasets()

ks_train = encode_launch_dt(ks_train)
ks_train = encode_deadline_dt(ks_train)

ks_test = encode_launch_dt(ks_test)
ks_test = encode_deadline_dt(ks_test)

relevant_columns = ['name', 'category', 'main_category', 'currency', 'goal', 'country', 'launched', 'deadline',
                     'launched_hour', 'launched_day', 'launched_month',
                     'deadline_hour', 'deadline_day', 'deadline_month']
x_train, x_test, y_train, y_test = get_xy_by_columns(ks_train, ks_test, relevant_columns)


In [80]:
x_train['len_name'] = x_train['name'].str.len()
x_test['len_name'] = x_test['name'].str.len()

In [81]:
x_train['num_words_name'] = x_train['name'].str.split().str.len()
x_test['num_words_name'] = x_test['name'].str.split().str.len()

In [82]:
x_train['goal_thousands'] = x_train['goal'].round(-3)
x_test['goal_thousands'] = x_test['goal'].round(-3)

In [83]:
x_train['launched_month_sin'] = np.sin(2*np.pi*x_train['launched_month']/12)
x_train['launched_month_cos'] = np.cos(2*np.pi*x_train['launched_month']/12)
x_test['launched_month_sin'] = np.sin(2*np.pi*x_test['launched_month']/12)
x_test['launched_month_cos'] = np.cos(2*np.pi*x_test['launched_month']/12)


In [84]:
x_train['launched_day_sin'] = np.sin(2*np.pi*x_train['launched_day']/24)
x_train['launched_day_cos'] = np.cos(2*np.pi*x_train['launched_day']/24)
x_test['launched_day_sin'] = np.sin(2*np.pi*x_test['launched_day']/24)
x_test['launched_day_cos'] = np.cos(2*np.pi*x_test['launched_day']/24)


In [85]:
x_train = x_train.drop(columns=['launched_month', 'launched_day', 'name', 'launched', 'deadline'])
x_test = x_test.drop(columns=['launched_month', 'launched_day', 'name', 'launched', 'deadline'])
x_train, x_test = catboost_encode(x_train, x_test)

In [86]:
cls = fit_evaluate(x_train, x_test, y_train, y_test)

AUC of ROC on train : 0.7511
AUC of ROC on validation : 0.7403
